# Notebook 01b — Exploratory Data Analysis
## AeroTwinML · Hyderabad vs Karachi

**Objective:** Understand AQI patterns, weather-AQI relationships, and differences between
Hyderabad and Karachi. This EDA drives feature engineering and model design decisions.

**Key questions:**
1. How do AQI distributions differ between cities?
2. What weather variables correlate most with AQI?
3. Are there seasonal/diurnal patterns?
4. What lag windows are most predictive?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils.config import get
from utils.storage import load_parquet

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

# Load multi-city data
DATA_DIR = Path(get('storage.data_dir', '../data'))
merged_path = DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet'

try:
    df = load_parquet(merged_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    print(f'Loaded {len(df)} rows, cities: {df["city"].unique().tolist() if "city" in df.columns else ["single"]}')
except FileNotFoundError:
    print('No merged data found. Run notebook 01 first.')
    df = pd.DataFrame()

## 1. AQI Distribution by City

In [ ]:
if not df.empty and 'aqi' in df.columns:
    aqi_col = 'aqi'
elif not df.empty and 'om_forecast_aqi' in df.columns:
    aqi_col = 'om_forecast_aqi'
else:
    aqi_col = None

if aqi_col and 'city' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Histogram
    for city in df['city'].unique():
        city_data = df[df['city'] == city][aqi_col].dropna()
        axes[0].hist(city_data, bins=50, alpha=0.6, label=city, density=True)
    axes[0].set_title(f'{aqi_col} Distribution by City')
    axes[0].set_xlabel('AQI')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[0].grid(True, alpha=0.2)
    
    # Box plot
    df.boxplot(column=aqi_col, by='city', ax=axes[1])
    axes[1].set_title(f'{aqi_col} by City')
    axes[1].set_xlabel('City')
    axes[1].set_ylabel('AQI')
    plt.suptitle('')
    
    plt.tight_layout()
    plt.show()
    
    # Stats
    print('\nAQI Statistics by City:')
    print(df.groupby('city')[aqi_col].describe().round(1).to_string())

## 2. Weather vs AQI Correlation

In [ ]:
if aqi_col and not df.empty:
    weather_cols = ['temperature_2m', 'relative_humidity_2m', 'dew_point_2m',
                    'pressure_msl', 'wind_speed_10m', 'precipitation', 'cloud_cover']
    available = [c for c in weather_cols if c in df.columns]
    
    if available:
        # Per-city correlation
        fig, axes = plt.subplots(1, len(df['city'].unique()), figsize=(8*len(df['city'].unique()), 7))
        if len(df['city'].unique()) == 1:
            axes = [axes]
        
        for ax, city in zip(axes, df['city'].unique()):
            city_df = df[df['city'] == city]
            corr_cols = [aqi_col] + available
            corr = city_df[corr_cols].corr()
            sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
            ax.set_title(f'{city} — AQI vs Weather Correlation')
        
        plt.tight_layout()
        plt.show()

## 3. Diurnal Pattern (AQI by Hour of Day)

In [ ]:
if aqi_col and 'city' in df.columns:
    df['hour'] = df['timestamp'].dt.hour
    
    fig, ax = plt.subplots(figsize=(14, 6))
    for city in df['city'].unique():
        hourly = df[df['city'] == city].groupby('hour')[aqi_col].mean()
        ax.plot(hourly.index, hourly.values, marker='o', label=city, linewidth=2)
    
    ax.set_title('Average AQI by Hour of Day', fontsize=14)
    ax.set_xlabel('Hour')
    ax.set_ylabel('Average AQI')
    ax.legend()
    ax.grid(True, alpha=0.2)
    ax.set_xticks(range(24))
    plt.show()

## 4. Lag Autocorrelation Analysis

This tells us how far ahead we can meaningfully forecast.

In [ ]:
if aqi_col:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for ax, city in zip(axes, df['city'].unique()):
        city_series = df[df['city'] == city][aqi_col].dropna()
        if len(city_series) < 100:
            ax.text(0.5, 0.5, f'{city}: insufficient data', ha='center', va='center', transform=ax.transAxes)
            continue
        
        max_lag = min(96, len(city_series) // 2)
        lags = range(1, max_lag + 1)
        autocorr = [city_series.autocorr(lag=lag) for lag in lags]
        
        ax.bar(lags, autocorr, color='#00b4d8', alpha=0.7)
        ax.axhline(y=0, color='white', linestyle='--', linewidth=0.5)
        for h, color in [(24, '#ff7e00'), (48, '#ff0000'), (72, '#8f3f97')]:
            ax.axvline(x=h, color=color, linestyle='--', linewidth=1, label=f'{h}h')
        ax.set_title(f'{city} — AQI Autocorrelation')
        ax.set_xlabel('Lag (hours)')
        ax.set_ylabel('Autocorrelation')
        ax.legend()
        ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.show()

## 5. Seasonal Patterns

In [ ]:
if aqi_col and 'city' in df.columns:
    df['month'] = df['timestamp'].dt.month
    
    fig, ax = plt.subplots(figsize=(14, 6))
    for city in df['city'].unique():
        monthly = df[df['city'] == city].groupby('month')[aqi_col].mean()
        ax.plot(monthly.index, monthly.values, marker='o', label=city, linewidth=2)
    
    ax.set_title('Average AQI by Month', fontsize=14)
    ax.set_xlabel('Month')
    ax.set_ylabel('Average AQI')
    ax.legend()
    ax.grid(True, alpha=0.2)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    plt.show()

## 6. Missing Data Analysis

In [ ]:
if not df.empty:
    for city in df['city'].unique():
        city_df = df[df['city'] == city]
        missing = city_df.isnull().sum()
        missing_pct = (missing / len(city_df) * 100).round(1)
        missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
        missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)
        
        print(f'\n=== {city} ({len(city_df)} rows) ===')
        if not missing_df.empty:
            print(missing_df.to_string())
        else:
            print('No missing values')

## Summary

Key findings:
- **City differences:** Karachi and Hyderabad have different AQI distributions and weather patterns
- **Weather correlation:** Temperature, humidity, and wind speed correlate with AQI
- **Lag structure:** Autocorrelation at 24h/48h/72h horizons shows predictive signal
- **Diurnal patterns:** AQI varies by hour of day (traffic, industrial activity)
- **Seasonal patterns:** AQI varies by month (monsoon, winter inversions)

**Next:** Notebook 02 — Feature Engineering